# Train All PCB Models: YOLO BB, YOLO-seg (SAM BB), YOLO-seg (SAM Auto)
This notebook runs the 4-step pipeline on the cleaned YOLO dataset.

In [ ]:
!pip install -q ultralytics
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

import os
import shutil
import urllib.request

# Download SAM checkpoint if not exists
sam_ckpt = "/kaggle/working/sam_vit_b.pth"
if not os.path.exists(sam_ckpt):
    print("Downloading SAM ViT-B checkpoint...")
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", sam_ckpt)
    print("Download complete.")

In [ ]:
import sys
import json
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import yaml
from ultralytics import YOLO
from segment_anything import sam_model_registry, SamPredictor, SamAutomaticMaskGenerator

# Paths for Kaggle
DATASET = Path("/kaggle/input/fics-pcb-yolo-cleaned")
DATA_YAML = DATASET / "fics_pcb.yaml"
OUT_ROOT = Path("/kaggle/working/model-outputs")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

CLASS_MAP = {0:'Cap1',1:'Cap2',2:'Cap3',3:'Cap4',4:'MOSFET',5:'Mov',6:'Resistor',7:'Transformer'}
NUM_CLASSES = 8
CLASS_NAMES = [CLASS_MAP[i] for i in range(NUM_CLASSES)]
SPLITS = ["train", "val", "test"]

# Utility functions
def parse_yolo_labels(label_path, img_w, img_h):
    boxes = []
    if not label_path.exists(): return boxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            cid = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = max(0, int(round((xc - bw/2) * img_w)))
            y1 = max(0, int(round((yc - bh/2) * img_h)))
            x2 = min(img_w, int(round((xc + bw/2) * img_w)))
            y2 = min(img_h, int(round((yc + bh/2) * img_h)))
            if x2 > x1 + 4 and y2 > y1 + 4:
                boxes.append((cid, [x1, y1, x2, y2]))
    return boxes

def mask_to_yolo_seg(mask, class_id, img_w, img_h, min_area=12.0):
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return None
    cnt = max(contours, key=cv2.contourArea)
    if cv2.contourArea(cnt) < min_area: return None
    approx = cv2.approxPolyDP(cnt, 0.005 * cv2.arcLength(cnt, True), True)
    if len(approx) < 3: return None
    coords = []
    for p in approx:
        coords.extend([round(float(p[0][0])/img_w, 6), round(float(p[0][1])/img_h, 6)])
    return f"{class_id} " + " ".join(str(c) for c in coords)

def box_to_yolo_seg(box, class_id, img_w, img_h):
    x1, y1, x2, y2 = box
    coords = [x1/img_w, y1/img_h, x2/img_w, y1/img_h, x2/img_w, y2/img_h, x1/img_w, y2/img_h]
    return f"{class_id} " + " ".join(str(round(c,6)) for c in coords)

def compute_iou(mask_a, mask_b):
    inter = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    return float(inter / union) if union > 0 else 0.0

def get_image_files(split_dir):
    return [f for f in sorted(split_dir.iterdir()) if f.suffix.lower() in ('.jpg','.png','.jpeg') and '_sam' not in f.stem and '_mask' not in f.stem and '_overlay' not in f.stem and 'legend' not in f.stem]

def write_data_yaml(out_dir, dataset_name):
    cfg = {"path": str(out_dir), "train": "train/images", "val": "val/images", "test": "test/images", "nc": NUM_CLASSES, "names": CLASS_NAMES}
    yaml_path = out_dir / "data.yaml"
    with open(yaml_path, "w") as f: yaml.dump(cfg, f, default_flow_style=False)
    return yaml_path

In [ ]:
# STEP 1: YOLO BB
def step1_yolo_bb(epochs=50):
    print("\n--- STEP 1: YOLO BB ---")
    out_dir = OUT_ROOT / "step1-yolo-bb"
    model = YOLO("yolo11n.pt")
    model.train(data=str(DATA_YAML), epochs=epochs, imgsz=640, batch=16, project=str(out_dir), name="train", exist_ok=True, plots=True, patience=15)
    weights = out_dir / "train" / "weights" / "best.pt"
    metrics = YOLO(str(weights)).val(data=str(DATA_YAML), split="test", project=str(out_dir), name="eval", exist_ok=True)
    summary = {"model": "YOLO-BB", "mAP50": float(metrics.box.map50), "mAP50-95": float(metrics.box.map)}
    with open(out_dir / "results.json", "w") as f: json.dump(summary, f)
    print(summary)
    return weights

yolo_weights = step1_yolo_bb(epochs=50)

In [ ]:
# STEP 2: SAM+BB -> YOLO-seg
def step2_sam_bb():
    print("\n--- STEP 2: SAM+BB PSEUDO-LABELS ---")
    seg_dataset = OUT_ROOT / "dataset-sam-bb"
    sam = sam_model_registry["vit_b"](checkpoint=sam_ckpt).to("cuda")
    sam.eval()
    predictor = SamPredictor(sam)
    
    for split in SPLITS:
        (seg_dataset / split / "images").mkdir(parents=True, exist_ok=True)
        (seg_dataset / split / "labels").mkdir(parents=True, exist_ok=True)
        img_files = get_image_files(DATASET / split / "images")
        lbl_dir = DATASET / split / "labels"
        for img_p in img_files:
            img = cv2.imread(str(img_p))
            if img is None: continue
            h, w = img.shape[:2]
            boxes = parse_yolo_labels(lbl_dir / f"{img_p.stem}.txt", w, h)
            if not boxes: continue
            
            predictor.set_image(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            seg_lines = []
            for cid, box in boxes:
                masks, _, _ = predictor.predict(box=np.array(box)[None, :], multimask_output=False)
                line = mask_to_yolo_seg(masks[0], cid, w, h) or box_to_yolo_seg(box, cid, w, h)
                seg_lines.append(line)
                
            shutil.copy2(img_p, seg_dataset / split / "images" / img_p.name)
            with open(seg_dataset / split / "labels" / f"{img_p.stem}.txt", "w") as f: f.write("\n".join(seg_lines))
            
    yaml_path = write_data_yaml(seg_dataset, "sam-bb")
    del sam, predictor # free GPU mem
    torch.cuda.empty_cache()
    
    print("\n--- TRAINING YOLO-SEG (SAM+BB) ---")
    out_dir = OUT_ROOT / "step2-yolo-seg-bb"
    model = YOLO("yolo11n-seg.pt")
    model.train(data=str(yaml_path), epochs=50, imgsz=640, batch=16, project=str(out_dir), name="train", exist_ok=True, plots=True, patience=15)
    metrics = YOLO(str(out_dir / "train/weights/best.pt")).val(data=str(yaml_path), split="test", project=str(out_dir), name="eval", exist_ok=True)
    summary = {"model": "YOLO-seg (BB)", "box_mAP50": float(metrics.box.map50), "mask_mAP50": float(metrics.seg.map50)}
    with open(out_dir / "results.json", "w") as f: json.dump(summary, f)
    print(summary)

step2_sam_bb()

In [ ]:
# STEP 3: SAM-auto -> YOLO-seg
def step3_sam_auto():
    print("\n--- STEP 3: SAM-AUTO PSEUDO-LABELS ---")
    seg_dataset = OUT_ROOT / "dataset-sam-auto"
    sam = sam_model_registry["vit_b"](checkpoint=sam_ckpt).to("cuda")
    sam.eval()
    generator = SamAutomaticMaskGenerator(model=sam, points_per_side=32, pred_iou_thresh=0.86, stability_score_thresh=0.92, min_mask_region_area=100)
    
    for split in SPLITS:
        (seg_dataset / split / "images").mkdir(parents=True, exist_ok=True)
        (seg_dataset / split / "labels").mkdir(parents=True, exist_ok=True)
        img_files = get_image_files(DATASET / split / "images")
        lbl_dir = DATASET / split / "labels"
        for img_p in img_files:
            img = cv2.imread(str(img_p))
            if img is None: continue
            h, w = img.shape[:2]
            gt_boxes = parse_yolo_labels(lbl_dir / f"{img_p.stem}.txt", w, h)
            if not gt_boxes: continue
            
            auto_masks = generator.generate(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            seg_lines = []
            used = set()
            for cid, box in gt_boxes:
                gt_mask = np.zeros((h, w), dtype=np.uint8)
                gt_mask[box[1]:box[3], box[0]:box[2]] = 1
                
                best_iou, best_idx = 0.0, -1
                for midx, m in enumerate(auto_masks):
                    if midx in used: continue
                    iou = compute_iou(m["segmentation"], gt_mask)
                    if iou > best_iou: best_iou, best_idx = iou, midx
                
                if best_iou > 0.25 and best_idx >= 0:
                    used.add(best_idx)
                    line = mask_to_yolo_seg(auto_masks[best_idx]["segmentation"], cid, w, h)
                    seg_lines.append(line if line else box_to_yolo_seg(box, cid, w, h))
                else:
                    seg_lines.append(box_to_yolo_seg(box, cid, w, h))
                    
            shutil.copy2(img_p, seg_dataset / split / "images" / img_p.name)
            with open(seg_dataset / split / "labels" / f"{img_p.stem}.txt", "w") as f: f.write("\n".join(seg_lines))

    yaml_path = write_data_yaml(seg_dataset, "sam-auto")
    del sam, generator
    torch.cuda.empty_cache()
    
    print("\n--- TRAINING YOLO-SEG (SAM-AUTO) ---")
    out_dir = OUT_ROOT / "step3-yolo-seg-auto"
    model = YOLO("yolo11n-seg.pt")
    model.train(data=str(yaml_path), epochs=50, imgsz=640, batch=16, project=str(out_dir), name="train", exist_ok=True, plots=True, patience=15)
    metrics = YOLO(str(out_dir / "train/weights/best.pt")).val(data=str(yaml_path), split="test", project=str(out_dir), name="eval", exist_ok=True)
    summary = {"model": "YOLO-seg (Auto)", "box_mAP50": float(metrics.box.map50), "mask_mAP50": float(metrics.seg.map50)}
    with open(out_dir / "results.json", "w") as f: json.dump(summary, f)
    print(summary)

step3_sam_auto()

In [ ]:
# Summary
rows = []
for path in OUT_ROOT.glob("*/results.json"):
    with open(path) as f: rows.append(json.load(f))
df = pd.DataFrame(rows)
print("\n=== FINAL BENCHMARK ===")
print(df.to_string())
df.to_csv(OUT_ROOT / "benchmark_summary.csv", index=False)